# Week 3 · Notebook 2 — Visualizing the A–Z Handwritten Dataset
**CSCI 250 · Fall 2026**

The **A–Z Handwritten** dataset: each row is one character — the **first column is a label** (0–25 → A–Z) and the remaining **784 columns** are the pixels of a 28×28 grayscale image. We load with Pandas, reshape with NumPy (Week 2!), and display with Matplotlib.

## 0. Setup

In [ ]:
!pip -q install pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Get the data
The full `A_Z-HandwrittenData.csv` (from Canvas) is large. To keep this notebook fast and self-contained we **synthesize a small, balanced sample** with the same structure (1 label column + 784 pixel columns). To use the **real** file instead, upload it and set `USE_REAL = True`.

In [ ]:
USE_REAL = False   # set True after uploading A_Z-HandwrittenData.csv

if USE_REAL:
    df = pd.read_csv('A_Z-HandwrittenData.csv', header=None)
else:
    rng = np.random.default_rng(0)
    rows = []
    for label in range(26):                 # A..Z
        n = rng.integers(20, 60)             # uneven counts on purpose
        for _ in range(n):
            img = np.zeros((28, 28))
            # draw a faint blob + the letter index as brightness, so letters differ
            cx, cy = rng.integers(8, 20, size=2)
            img[cy-3:cy+3, cx-3:cx+3] = 80 + label * 6 + rng.integers(0, 40)
            img += rng.integers(0, 25, size=(28, 28))
            rows.append([label] + img.clip(0, 255).reshape(-1).tolist())
    df = pd.DataFrame(rows)
print('shape:', df.shape)   # (n_samples, 785)
df.iloc[:3, :6]

## 2. Split labels and pixels

In [ ]:
labels = df.iloc[:, 0].values.astype(int)   # 0..25
pixels = df.iloc[:, 1:].values.astype(float) # (n, 784)
print('labels:', labels.shape, ' pixels:', pixels.shape)
letter = lambda i: chr(ord('A') + int(i))

## 3. Show ONE character
Reshape its 784 pixels back to 28×28 (Week 2 `reshape`).

In [ ]:
img = pixels[0].reshape(28, 28)
plt.figure(figsize=(3, 3))
plt.imshow(img, cmap='gray')
plt.title('label = ' + letter(labels[0]))
plt.axis('off')
plt.show()

## 4. A 3×3 grid of characters

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, idx in zip(axes.ravel(), range(9)):
    ax.imshow(pixels[idx].reshape(28, 28), cmap='gray')
    ax.set_title(letter(labels[idx]))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Bar chart — how many examples per letter?
Is the dataset **balanced**? This is the key question for A3.

In [ ]:
counts = pd.Series(labels).value_counts().sort_index()
plt.figure(figsize=(10, 4))
plt.bar([letter(i) for i in counts.index], counts.values, color='teal')
plt.xlabel('letter'); plt.ylabel('number of examples')
plt.title('Examples per letter')
plt.show()

## 6. Histogram — distribution of average brightness per image

In [ ]:
brightness = pixels.mean(axis=1)   # Week-2 axis op: mean per row
plt.figure(figsize=(8, 4))
plt.hist(brightness, bins=30, color='steelblue', edgecolor='white')
plt.xlabel('average pixel brightness'); plt.ylabel('count of images')
plt.title('How bright is a typical character?')
plt.show()

## 7. Scatter — does brightness relate to the letter index?
A quick scatter to look for any relationship between two numbers.

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(labels, brightness, alpha=0.3)
plt.xlabel('label (0=A .. 25=Z)'); plt.ylabel('avg brightness')
plt.title('Brightness vs letter')
plt.show()

## 8. Your turn (for A3)
1. Print the **mean brightness per letter** using a Pandas `groupby`.
2. Which letter has the **most** examples and which the **fewest**?
3. In 100–150 words: is this dataset balanced, and why would imbalance matter when we train a classifier in a few weeks?

In [ ]:
# Hint for #1:
# tmp = pd.DataFrame({'label': labels, 'brightness': brightness})
# print(tmp.groupby('label')['brightness'].mean())


## 9. (Optional) Ask an LLM for another chart idea
Use **Claude** or **Gemini** to suggest one more chart that would reveal something about this dataset, then build it. Never paste an API key — use Colab Secrets.

In [ ]:
# Optional sidebar — uncomment and add your key via Colab Secrets (userdata.get).
# import os
# import google.generativeai as genai
# from google.colab import userdata
# genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
# model = genai.GenerativeModel('gemini-2.5-flash')
# resp = model.generate_content(
#     'I have a handwritten-letters dataset (label + 784 pixel columns). '
#     'Suggest one Matplotlib chart that reveals data quality issues.')
# print(resp.text)
